# Single Crystal Data Reduction Workflow for MAGiC diffractometer

This notebook presents skeleton of data reduction procedure for Single Crystal for McStas h5 file of MAGiC using SCIPP. The steps are described on the corresponding [confluence page](https://confluence.ess.eu/spaces/~iuriikibalin/pages/788704198/Single-Crystal+Data+Reduction+Workflow+for+MAGiC+with+Scipp+Tasks).


In [1]:
%matplotlib widget

from IPython.display import display

from ui import *
from state import STATE
from callbacks import *


            
# find_peaks_button.on_click(run_peak_finder)
find_peaks_hist_button.on_click(run_peak_finder_hist)
integration_button.on_click(run_peak_integration)
load_events_button.on_click(load_file)
display_button.on_click(display_data)
display_da_button.on_click(display_da_data)
indexation_button.on_click(indexate_peaks)
hide_output_data_button.on_click(lambda x: output_data.clear_output())    
display_peaks_button.on_click(display_peaks)
hide_peaks_button.on_click(lambda x: peaks_output.clear_output())   

# ---------------------------------------------------------
# Display UI
# ---------------------------------------------------------
ui_load_data = widgets.VBox([    fc, fc_vanadium, 
    load_events_button, 
    ui_control_display, 
    ui_da_data,
    output_data, 
],
    layout=widgets.Layout(align_items='center'))

ui_find_peaks = widgets.VBox([   
    norm_rbuttons, 
    # find_peaks_button, 
    find_peaks_hist_button,
    ui_peaks_button, 
    peaks_output, 
],
    layout=widgets.Layout(align_items='center'))


ui_indexation = widgets.VBox([   
    ui_abc, 
    ui_angles, 
    indexation_button, 
    indexation_output
],
    layout=widgets.Layout(align_items='center'))


ui_integration = widgets.VBox([   
    integration_button, 
    integration_output
],
    layout=widgets.Layout(align_items='center'))

accordion = widgets.Accordion(
    children=[ui_load_data, ui_find_peaks, ui_indexation, ui_integration], 
    titles=('Load Data', 'Find Peaks', 'Indexation', 'Integration',))

accordion.layout = widgets.Layout(width='100%')
# intro_html = widgets.HTML(
#     value="""
#     <div style="text-align:center;">
#         <img src="scheme.png" style="max-width:70%; height:auto;">
#     </div>
#     """
# )
intro_html = widgets.HTML("""
<div style="text-align:center;">
    <img src="scheme.png" style="max-width:100%; height:auto;"><br>
</div>
""")
display(intro_html)
display_center(
    accordion,
)


HTML(value='\n<div style="text-align:center;">\n    <img src="scheme.png" style="max-width:100%; height:auto;"…

In [ ]:
from IPython.utils.capture import capture_output

import ast
from IPython.display import display

def execute_with_last_expr(code, global_ns):
    """
    Executes code and returns the value of the last expression (if any).
    """
    tree = ast.parse(code)

    # If last node is an expression, separate it
    if isinstance(tree.body[-1], ast.Expr):
        last_expr = ast.Expression(tree.body[-1].value)
        exec(compile(ast.Module(tree.body[:-1], []), "<cell>", "exec"), global_ns)
        return eval(compile(last_expr, "<cell>", "eval"), global_ns)
    else:
        exec(code, global_ns)
        return None
    
# Global namespace shared with the rest of your application
EXEC_GLOBALS = globals()


def make_code_cell(initial_text=""):
    """
    Creates a mini notebook-like code cell:
    - Text area for Python code
    - Run button
    - Output area
    """

    code_area = widgets.Textarea(
        value=initial_text,
        placeholder="Write Python code here...\n\n Global variables: instrument, diag.",
        layout=widgets.Layout(width="100%", height="200px"),
        style={"font_family": "monospace"}
    )

    interrupt_requested = {"value": False}

    run_button = widgets.Button(
        description="Run",
        button_style="success",
        tooltip="Execute the code",
        icon="play"
    )

    interrupt_button = widgets.Button(
        description="Interrupt",
        button_style="danger",
        tooltip="Interrupt execution",
        icon="stop",
        disabled=True,
    )

    output = widgets.Output(layout=widgets.Layout(border="1px solid #ccc", padding='6px'))

    def interrupt_execution(_):
        interrupt_requested["value"] = True
        output.append_stdout("Interrupt requested...")
        interrupt_button.disabled = True
        if run_button.disabled:
            run_button.icon = "play"
            run_button.button_style = "success"
            run_button.disabled = False

    def run_code(_):
        output.clear_output()
        run_button.disabled = True
        run_button.button_style = "warning"
        run_button.icon = "spinner"
        interrupt_button.disabled = False
        interrupt_requested["value"] = False

        code = code_area.value

        def trace_func(frame, event, arg):
            if interrupt_requested["value"]:
                raise KeyboardInterrupt("Execution interrupted by user")
            return trace_func

        def target():
            try:
                # sys.settrace(trace_func)
                with capture_output() as cap:
                    # exec(code, EXEC_GLOBALS)
                    result = execute_with_last_expr(code, EXEC_GLOBALS)


                # Send captured output to widget
                with output:
                    if cap.stdout:
                        print(cap.stdout, end="")
                    if cap.stderr:
                        print(cap.stderr, end="")
                
                    if result is not None:
                        display(result)

            except KeyboardInterrupt:
                with output:
                    print("Execution interrupted by user.")

            except Exception:
                with output:
                    traceback.print_exc()

            finally:
                sys.settrace(None)
                run_button.disabled = False
                run_button.button_style = "success"
                run_button.icon = "play"
                interrupt_button.disabled = True


        thread = threading.Thread(target=target, daemon=True)
        thread.start()

    interrupt_button.on_click(interrupt_execution)
    run_button.on_click(run_code)

    return widgets.VBox([
        widgets.HTML("<b>Python Code Cell</b>"),
        code_area,
        widgets.HBox([run_button, interrupt_button]),
        output
    ])

code_cell = make_code_cell()
display(code_cell)